# USDA `fdc_id` coverage

How many distinct `fdc_id` values appear in `food`, `food_nutrient`, `food_portion`, and `input_food`?

The **universe** of all foods is `usda.food.fdc_id` (primary key). Other tables reference foods via `fdc_id` or, for recipe inputs, `fdc_id_of_input_food`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "scratch":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

from db import connect

In [17]:
FDC_COVERAGE_SQL = """
WITH universe AS (
    SELECT COUNT(DISTINCT fdc_id) AS total_fdc_ids FROM usda.food
),
stats AS (
    SELECT
        'food' AS table_name,
        'fdc_id (primary key)' AS how_fdc_id_is_used,
        COUNT(DISTINCT fdc_id) AS distinct_fdc_ids,
        COUNT(*) AS row_count
    FROM usda.food

    UNION ALL

    SELECT
        'food_nutrient',
        'fdc_id (food with nutrient rows)',
        COUNT(DISTINCT fdc_id),
        COUNT(*)
    FROM usda.food_nutrient

    UNION ALL

    SELECT
        'food_portion',
        'fdc_id (food with portion rows)',
        COUNT(DISTINCT fdc_id),
        COUNT(*)
    FROM usda.food_portion

    UNION ALL

    SELECT
        'input_food',
        'fdc_id (parent / composite food)',
        COUNT(DISTINCT fdc_id),
        COUNT(*)
    FROM usda.input_food

    UNION ALL

    SELECT
        'input_food',
        'fdc_id_of_input_food (ingredient link, when populated)',
        COUNT(DISTINCT fdc_id_of_input_food::bigint) FILTER (
            WHERE fdc_id_of_input_food IS NOT NULL
              AND TRIM(fdc_id_of_input_food) <> ''
              AND fdc_id_of_input_food ~ '^[0-9]+$'
        ),
        COUNT(*) FILTER (
            WHERE fdc_id_of_input_food IS NOT NULL
              AND TRIM(fdc_id_of_input_food) <> ''
              AND fdc_id_of_input_food ~ '^[0-9]+$'
        )
    FROM usda.input_food
)
SELECT
    s.table_name,
    s.how_fdc_id_is_used,
    s.distinct_fdc_ids,
    s.row_count,
    u.total_fdc_ids,
    ROUND(100.0 * s.distinct_fdc_ids / u.total_fdc_ids, 4) AS pct_of_all_fdc_ids
FROM stats s
CROSS JOIN universe u
ORDER BY s.table_name, s.how_fdc_id_is_used
"""

with connect() as conn:
    coverage = pd.read_sql(FDC_COVERAGE_SQL, conn)

coverage

/var/folders/qq/pmrcg6p90876k5_k2lxpk37w0000gn/T/ipykernel_58179/1063867807.py:70: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  coverage = pd.read_sql(FDC_COVERAGE_SQL, conn)


,table_name,how_fdc_id_is_used,distinct_fdc_ids,row_count,total_fdc_ids,pct_of_all_fdc_ids
0,food,fdc_id (primary key),2101279,2101279,2101279,100.0000
1,food_nutrient,fdc_id (food with nutrient rows),37457,827280,2101279,1.7826
2,food_portion,fdc_id (food with portion rows),15224,47064,2101279,0.7245
3,input_food,fdc_id (parent / composite food),5431,18584,2101279,0.2585
4,input_food,"fdc_id_of_input_food (ingredient link, when po...",0,0,2101279,0.0000


In [18]:
# Pivot for a compact view (one row per table for direct fdc_id columns)
direct = coverage[~coverage["how_fdc_id_is_used"].str.contains("fdc_id_of_input")].copy()
direct = direct.rename(columns={"how_fdc_id_is_used": "fdc_id_column"})
direct[["table_name", "fdc_id_column", "distinct_fdc_ids", "row_count", "total_fdc_ids", "pct_of_all_fdc_ids"]]

,table_name,fdc_id_column,distinct_fdc_ids,row_count,total_fdc_ids,pct_of_all_fdc_ids
0,food,fdc_id (primary key),2101279,2101279,2101279,100.0000
1,food_nutrient,fdc_id (food with nutrient rows),37457,827280,2101279,1.7826
2,food_portion,fdc_id (food with portion rows),15224,47064,2101279,0.7245
3,input_food,fdc_id (parent / composite food),5431,18584,2101279,0.2585


In [19]:
# Overlap: which foods appear in multiple extension tables?
OVERLAP_SQL = """
WITH u AS (SELECT fdc_id FROM usda.food),
     n AS (SELECT DISTINCT fdc_id FROM usda.food_nutrient),
     p AS (SELECT DISTINCT fdc_id FROM usda.food_portion),
     i AS (SELECT DISTINCT fdc_id FROM usda.input_food)
SELECT
    (SELECT COUNT(*) FROM u) AS all_foods,
    (SELECT COUNT(*) FROM n) AS in_food_nutrient,
    (SELECT COUNT(*) FROM p) AS in_food_portion,
    (SELECT COUNT(*) FROM i) AS in_input_food_parent,
    (SELECT COUNT(*) FROM n JOIN p USING (fdc_id)) AS nutrient_and_portion,
    (SELECT COUNT(*) FROM n JOIN i USING (fdc_id)) AS nutrient_and_input,
    (SELECT COUNT(*) FROM p JOIN i USING (fdc_id)) AS portion_and_input,
    (SELECT COUNT(*) FROM n JOIN p USING (fdc_id) JOIN i USING (fdc_id)) AS all_three_extensions
"""

with connect() as conn:
    overlap = pd.read_sql(OVERLAP_SQL, conn)

overlap.T.rename(columns={0: "count"})

/var/folders/qq/pmrcg6p90876k5_k2lxpk37w0000gn/T/ipykernel_58179/1250471309.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  overlap = pd.read_sql(OVERLAP_SQL, conn)


,count
all_foods,2101279
in_food_nutrient,37720
in_food_portion,15224
in_input_food_parent,5431
nutrient_and_portion,4601
nutrient_and_input,0
portion_and_input,5373
all_three_extensions,0


## Notes (Supabase section)

- **`food`** defines every `fdc_id` in this release (~2.1M). That set is the denominator for percentages above.
- **`food_nutrient`** and **`food_portion`** each store many rows per `fdc_id` (nutrients / serving sizes), so `row_count` ≫ `distinct_fdc_ids`.
- **`input_food`** links a **parent** `fdc_id` (composite dish) to ingredients. In this April 2026 extract, `fdc_id_of_input_food` is empty for all loaded rows; ingredients are identified via `sr_code` / `sr_description` instead.
- **Why `food_mvp` can show ~5,910 rows while CSV EDA shows ~13,043 with both:** the SQL in `sql/06_create_food_mvp.sql` matches the EXISTS logic below, but Supabase had only a **partial** `food_nutrient` load (~960k / ~27M rows). Until `scripts/load_food_nutrient_chunked.py` finishes, only ~39k foods have nutrient rows in the DB (mostly `sr_legacy_food`), so survey/FNDDS and foundation foods with portions never qualify. After a full reload, run `sql/07_refresh_food_mvp.sql` and expect counts aligned with the CSV section (~13k with both).

## Local CSV analysis (complete files, **excluding branded food**)

**Universe:** all `fdc_id` in `food.csv` where `data_type != 'branded_food'`.

Rows in `food_nutrient`, `food_portion`, and `input_food` are only counted if their `fdc_id` is in that universe (branded parents/ingredients are dropped).

In [20]:
from pathlib import Path

DATA_DIR = ROOT / "Data" / "All_Food_Data_April_2026"
CHUNK_SIZE = 500_000
BRANDED_TYPE = "branded_food"


def load_non_branded_universe() -> tuple[pd.DataFrame, set[int]]:
    """Return non-branded food rows and their fdc_id set."""
    food = pd.read_csv(
        DATA_DIR / "food.csv",
        usecols=["fdc_id", "data_type"],
        dtype={"fdc_id": "int64", "data_type": "string"},
    )
    non_branded = food.loc[food["data_type"] != BRANDED_TYPE]
    universe = set(non_branded["fdc_id"])
    return non_branded, universe


def parse_fdc_ids(series: pd.Series, universe: set[int]) -> pd.Series:
    ids = pd.to_numeric(series, errors="coerce").dropna().astype("int64")
    return ids[ids.isin(universe)]


def count_fdc_in_csv(path: Path, universe: set[int], column: str = "fdc_id") -> tuple[int, int]:
    """Distinct fdc_ids (in universe) and total rows read."""
    distinct: set[int] = set()
    rows = 0
    for chunk in pd.read_csv(path, usecols=[column], dtype={column: "string"}, chunksize=CHUNK_SIZE):
        rows += len(chunk)
        distinct.update(parse_fdc_ids(chunk[column], universe).unique())
    return len(distinct), rows


def count_input_ingredient_fdc(path: Path, universe: set[int]) -> tuple[int, int]:
    distinct: set[int] = set()
    rows = 0
    for chunk in pd.read_csv(
        path,
        usecols=["fdc_id_of_input_food"],
        dtype={"fdc_id_of_input_food": "string"},
        chunksize=CHUNK_SIZE,
    ):
        s = chunk["fdc_id_of_input_food"].astype("string").str.strip()
        valid = s.str.fullmatch(r"[0-9]+", na=False)
        sub = s[valid]
        rows += len(sub)
        if len(sub):
            ids = sub.astype("int64")
            distinct.update(ids[ids.isin(universe)].unique())
    return len(distinct), rows


def collect_fdc_ids(path: Path, universe: set[int]) -> set[int]:
    distinct: set[int] = set()
    for chunk in pd.read_csv(path, usecols=["fdc_id"], dtype={"fdc_id": "string"}, chunksize=CHUNK_SIZE):
        distinct.update(parse_fdc_ids(chunk["fdc_id"], universe).unique())
    return distinct

In [21]:
non_branded_food, universe_ids = load_non_branded_universe()
total_non_branded = len(universe_ids)
branded_count = int((pd.read_csv(DATA_DIR / "food.csv", usecols=["data_type"])["data_type"] == BRANDED_TYPE).sum())

stats = [
    {
        "table_name": "food",
        "how_fdc_id_is_used": "fdc_id (primary key, non-branded only)",
        "distinct_fdc_ids": total_non_branded,
        "row_count": len(non_branded_food),
    }
]

for table, filename, label in [
    ("food_nutrient", "food_nutrient.csv", "fdc_id (food with nutrient rows)"),
    ("food_portion", "food_portion.csv", "fdc_id (food with portion rows)"),
    ("input_food", "input_food.csv", "fdc_id (parent / composite food)"),
]:
    d, r = count_fdc_in_csv(DATA_DIR / filename, universe_ids)
    stats.append({"table_name": table, "how_fdc_id_is_used": label, "distinct_fdc_ids": d, "row_count": r})

d_ing, r_ing = count_input_ingredient_fdc(DATA_DIR / "input_food.csv", universe_ids)
stats.append({
    "table_name": "input_food",
    "how_fdc_id_is_used": "fdc_id_of_input_food (ingredient link, when populated)",
    "distinct_fdc_ids": d_ing,
    "row_count": r_ing,
})

csv_coverage = pd.DataFrame(stats)
csv_coverage["total_fdc_ids"] = total_non_branded
csv_coverage["pct_of_non_branded_fdc_ids"] = (
    100.0 * csv_coverage["distinct_fdc_ids"] / total_non_branded
).round(4)

print(f"Branded foods excluded: {branded_count:,}")
print(f"Non-branded universe: {total_non_branded:,}\n")
csv_coverage

Branded foods excluded: 1,999,950
Non-branded universe: 101,329



,table_name,how_fdc_id_is_used,distinct_fdc_ids,row_count,total_fdc_ids,pct_of_non_branded_fdc_ids
0,food,"fdc_id (primary key, non-branded only)",101329,101329,101329,100.0000
1,food_nutrient,fdc_id (food with nutrient rows),89260,27195013,101329,88.0893
2,food_portion,fdc_id (food with portion rows),15245,47446,101329,15.0451
3,input_food,fdc_id (parent / composite food),5431,18584,101329,5.3598
4,input_food,"fdc_id_of_input_food (ingredient link, when po...",0,0,101329,0.0000


In [22]:
nutrient_ids = collect_fdc_ids(DATA_DIR / "food_nutrient.csv", universe_ids)
portion_ids = collect_fdc_ids(DATA_DIR / "food_portion.csv", universe_ids)
input_parent_ids = collect_fdc_ids(DATA_DIR / "input_food.csv", universe_ids)

both_ids = nutrient_ids & portion_ids

csv_overlap = pd.DataFrame(
    {
        "metric": [
            "non_branded_foods",
            "in_food_nutrient",
            "in_food_portion",
            "in_input_food_parent",
            "nutrient_and_portion",
            "nutrient_only",
            "portion_only",
            "all_three_extensions",
        ],
        "count": [
            total_non_branded,
            len(nutrient_ids),
            len(portion_ids),
            len(input_parent_ids),
            len(both_ids),
            len(nutrient_ids - portion_ids),
            len(portion_ids - nutrient_ids),
            len(nutrient_ids & portion_ids & input_parent_ids),
        ],
    }
)
csv_overlap["pct_of_non_branded"] = (100.0 * csv_overlap["count"] / total_non_branded).round(4)
csv_overlap

,metric,count,pct_of_non_branded
0,non_branded_foods,101329,100.0000
1,in_food_nutrient,89260,88.0893
2,in_food_portion,15245,15.0451
3,in_input_food_parent,5431,5.3598
4,nutrient_and_portion,13043,12.8719
5,nutrient_only,76217,75.2174
6,portion_only,2202,2.1731
7,all_three_extensions,5394,5.3233


In [23]:
by_data_type = (
    non_branded_food.assign(
        has_nutrients=non_branded_food["fdc_id"].isin(nutrient_ids),
        has_portions=non_branded_food["fdc_id"].isin(portion_ids),
        has_both=non_branded_food["fdc_id"].isin(both_ids),
    )
    .groupby("data_type", dropna=False)
    .agg(
        foods=("fdc_id", "count"),
        with_nutrients=("has_nutrients", "sum"),
        with_portions=("has_portions", "sum"),
        with_both=("has_both", "sum"),
    )
    .assign(
        pct_with_nutrients=lambda d: (100 * d["with_nutrients"] / d["foods"]).round(2),
        pct_with_both=lambda d: (100 * d["with_both"] / d["foods"]).round(2),
    )
    .sort_values("foods", ascending=False)
)
by_data_type

,foods,with_nutrients,with_portions,with_both,pct_with_nutrients,pct_with_both
data_type,,,,,,
sub_sample_food,75055,74758,0,0,99.60,0.00
sr_legacy_food,7793,7793,7533,7533,100.00,96.66
market_acquistion,7577,0,2201,0,0.00,0.00
survey_fndds_food,5432,5431,5395,5394,99.98,99.30
sample_food,4079,0,0,0,0.00,0.00
agricultural_acquisition,810,810,0,0,100.00,0.00
foundation_food,469,468,116,116,99.79,24.73
experimental_food,114,0,0,0,0.00,0.00


### CSV takeaways (non-branded only)

| Question | Answer |
|----------|--------|
| How many foods in scope? | **101,329** (`food.csv` minus ~2M `branded_food`) |
| Have any `food_nutrient` rows? | **89,260** (**88.1%**) |
| Have any `food_portion` rows? | **15,245** (**15.0%**) |
| Have **both** nutrients and portions? | **13,043** (**12.9%**) — use this as the target for `food_mvp` after a full DB load |
| `input_food` parents | **5,431** (mostly `survey_fndds_food`); **`fdc_id_of_input_food` still empty** everywhere |

**By type (foods with both):** `sr_legacy_food` 7,533 · `survey_fndds_food` 5,394 · `foundation_food` 116 (disjoint types → **13,043** total).

**Reliability note:** Many non-branded foods have sparse nutrient rows (median 1 per food). Treat **`nutrient_and_portion`** (~13k) as the strongest set for combined nutrition + serving-size work.

## CSV `food_mvp` and portion modifiers (density inference)

Build the same population as `usda.food_mvp` from local CSVs (non-branded, both `food_nutrient` and `food_portion`), then classify `food_portion.modifier` into **volume**, **mass**, and **other** to estimate how many foods could support inferred density (`gram_weight` + a volume-style portion).

In [24]:
import re

# Rebuild MVP ids if earlier CSV cells were not run in this session.
if "both_ids" not in globals():
    non_branded_food, universe_ids = load_non_branded_universe()
    nutrient_ids = collect_fdc_ids(DATA_DIR / "food_nutrient.csv", universe_ids)
    portion_ids = collect_fdc_ids(DATA_DIR / "food_portion.csv", universe_ids)
    both_ids = nutrient_ids & portion_ids

food_mvp = non_branded_food.loc[non_branded_food["fdc_id"].isin(both_ids)].copy()
food_mvp = food_mvp.sort_values("fdc_id").reset_index(drop=True)

MVP_CSV = ROOT / "scratch" / "food_mvp.csv"
food_mvp.to_csv(MVP_CSV, index=False)

print(f"food_mvp: {len(food_mvp):,} foods (non-branded with nutrients + portions)")
print(f"Saved → {MVP_CSV}")
food_mvp.head()

food_mvp: 13,043 foods (non-branded with nutrients + portions)
Saved → /Users/danielcosta/Berkeley/Capstone/scratch/food_mvp.csv


,fdc_id,data_type
0,167512,sr_legacy_food
1,167513,sr_legacy_food
2,167514,sr_legacy_food
3,167515,sr_legacy_food
4,167516,sr_legacy_food


In [28]:
import sys

sys.path.insert(0, str(ROOT / "scripts"))
from usda_volume_units import MASS_PATTERN, VOLUME_PATTERN, classify_modifier_text, text_has_volume

MEASURE_UNITS = pd.read_csv(DATA_DIR / "measure_unit.csv", dtype={"id": "string", "name": "string"})
MU_BY_ID = dict(zip(MEASURE_UNITS["id"], MEASURE_UNITS["name"]))


def load_mvp_portions(mvp_ids: set[int], chunk_size: int = 100_000) -> pd.DataFrame:
    chunks = []
    usecols = [
        "fdc_id",
        "modifier",
        "measure_unit_id",
        "portion_description",
        "gram_weight",
        "amount",
    ]
    for chunk in pd.read_csv(
        DATA_DIR / "food_portion.csv",
        usecols=usecols,
        dtype="string",
        chunksize=chunk_size,
    ):
        chunk["fdc_id"] = pd.to_numeric(chunk["fdc_id"], errors="coerce")
        chunk = chunk[chunk["fdc_id"].isin(mvp_ids)]
        if chunk.empty:
            continue
        chunk["modifier"] = chunk["modifier"].fillna("").str.strip()
        chunk["portion_description"] = chunk["portion_description"].fillna("").str.strip()
        chunk["measure_unit_name"] = chunk["measure_unit_id"].map(MU_BY_ID).fillna("")
        chunk["modifier_unit_group"] = chunk["modifier"].map(classify_modifier_text)
        chunk["gram_weight"] = pd.to_numeric(chunk["gram_weight"], errors="coerce")
        chunk["has_gram_weight"] = chunk["gram_weight"].notna() & (chunk["gram_weight"] > 0)
        chunk["has_volume_measure"] = chunk.apply(
            lambda r: text_has_volume(
                r["modifier"], r["portion_description"], r["measure_unit_name"]
            ),
            axis=1,
        )
        chunk["density_inferable"] = chunk["has_gram_weight"] & chunk["has_volume_measure"]
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)


mvp_portions = load_mvp_portions(both_ids)
print(f"Portion rows for food_mvp: {len(mvp_portions):,}")
print(f"Distinct modifiers: {mvp_portions['modifier'].nunique():,}")
print("cup/cups word-boundary check (should be 0):", mvp_portions["portion_description"].str.contains(r"cupcake", case=False, na=False).sum(), "rows mention cupcake")
mvp_portions.head()

Portion rows for food_mvp: 36,679
Distinct modifiers: 3,041
cup/cups word-boundary check (should be 0): 28 rows mention cupcake


,fdc_id,amount,measure_unit_id,portion_description,modifier,gram_weight,measure_unit_name,modifier_unit_group,has_gram_weight,has_volume_measure,density_inferable
0,167512,1.0,9999,,serving,34.0,undetermined,other,True,False,False
1,167513,1.0,9999,,serving 1 roll with icing,44.0,undetermined,other,True,False,False
2,167514,1.0,9999,,serving,28.0,undetermined,other,True,False,False
3,167515,1.0,9999,,serving,57.0,undetermined,other,True,False,False
4,167516,1.0,9999,,"waffle, square",39.0,undetermined,other,True,False,False


In [26]:
# Distinct modifiers among food_mvp portions, grouped by unit type (modifier text only)
modifier_catalog = (
    mvp_portions.groupby(["modifier", "modifier_unit_group"], dropna=False)
    .agg(
        portion_rows=("fdc_id", "count"),
        foods=("fdc_id", "nunique"),
        with_gram_weight=("has_gram_weight", "sum"),
        with_volume_context=("has_volume_measure", "sum"),
    )
    .reset_index()
    .sort_values(["modifier_unit_group", "portion_rows"], ascending=[True, False])
)

print("Modifiers by group (text classification):")
display(modifier_catalog.groupby("modifier_unit_group").agg(
    distinct_modifiers=("modifier", "nunique"),
    portion_rows=("portion_rows", "sum"),
    foods=("foods", "max"),
))

# Full list (scroll in dataframe); top examples per group
for group in ["volume", "mass", "other"]:
    print(f"\n--- {group} (top 25 by portion_rows) ---")
    display(
        modifier_catalog.loc[modifier_catalog["modifier_unit_group"] == group]
        .head(25)[["modifier", "portion_rows", "foods", "with_volume_context"]]
    )

modifier_catalog

Modifiers by group (text classification):


,distinct_modifiers,portion_rows,foods
modifier_unit_group,,,
mass,486,5141,3041
other,2294,27622,5325
volume,261,3916,1643



--- volume (top 25 by portion_rows) ---


,modifier,portion_rows,foods,with_volume_context
1511,cup,1691,1643,1691
2920,tbsp,548,548,548
2962,tsp,171,163,171
1514,cup (1 NLEA serving),98,98,98
2909,tablespoon,91,91,91
1625,"cup, chopped",71,71,71
1602,cup slices,68,67,68
1627,"cup, chopped or diced",58,58,58
1672,"cup, shredded",52,52,52
1508,cubic inch,51,51,51



--- mass (top 25 by portion_rows) ---


,modifier,portion_rows,foods,with_volume_context
2106,oz,3166,3041,0
1985,lb,281,281,0
2359,"piece, cooked, excluding refuse (yield from 1 ...",212,212,0
2983,unit (yield from 1 lb ready-to-cook chicken),106,106,0
2173,package (10 oz),50,37,0
1925,jar Gerber Second Food (4 oz),36,36,0
2123,oz (3 oz),35,35,0
2991,"unit, cooked (yield from 1 lb raw meat)",33,33,0
2174,package (10 oz) yields,32,30,0
2507,serving ( 3 oz ),30,30,0



--- other (top 25 by portion_rows) ---


,modifier,portion_rows,foods,with_volume_context
1138,90000,5325,5325,0
36,10205,3055,3055,3055
1787,fl oz,492,440,492
55,30000,463,463,463
285,60919,406,406,0
76,51000,374,374,374
51,21000,310,310,310
2838,steak,280,280,0
523,62015,265,265,0
2506,serving,265,265,0


,modifier,modifier_unit_group,portion_rows,foods,with_gram_weight,with_volume_context
2106,oz,mass,3166,3041,3166,0
1985,lb,mass,281,281,281,0
2359,"piece, cooked, excluding refuse (yield from 1 ...",mass,212,212,212,0
2983,unit (yield from 1 lb ready-to-cook chicken),mass,106,106,106,0
2173,package (10 oz),mass,50,37,50,0
...,...,...,...,...,...,...
2967,tsp or 1 packet,volume,1,1,1,1
2968,tsp packed,volume,1,1,1,1
2969,tsp rounded,volume,1,1,1,1
2970,tsp unpacked,volume,1,1,1,1


In [29]:
# --- How the density-inferable group is defined ---

def volume_in_modifier(mod: str) -> bool:
    return text_has_volume(mod, "", "")


def volume_in_description(desc: str) -> bool:
    return text_has_volume("", desc, "")


def volume_in_measure_unit(mu: str) -> bool:
    return text_has_volume("", "", mu)


inferable_portions = mvp_portions.loc[mvp_portions["density_inferable"]].copy()
inferable_portions["vol_modifier"] = inferable_portions.apply(
    lambda r: volume_in_modifier(r["modifier"]), axis=1
)
inferable_portions["vol_description"] = inferable_portions.apply(
    lambda r: volume_in_description(r["portion_description"]), axis=1
)
inferable_portions["vol_measure_unit"] = inferable_portions.apply(
    lambda r: volume_in_measure_unit(r["measure_unit_name"]), axis=1
)
inferable_portions["cup_modifier_exact"] = inferable_portions["modifier"].str.lower().isin(
    ["cup", "cups"]
)

food_signals = (
    inferable_portions.groupby("fdc_id", as_index=False)
    .agg(
        vol_modifier=("vol_modifier", "any"),
        vol_description=("vol_description", "any"),
        vol_measure_unit=("vol_measure_unit", "any"),
        cup_modifier_exact=("cup_modifier_exact", "any"),
        inferable_portion_rows=("fdc_id", "count"),
    )
)

criteria_steps = pd.DataFrame(
    {
        "step": ["1. Universe", "2. food_mvp", "3. Portion row", "4. Volume token", "5. Result"],
        "rule": [
            "food.csv, data_type != branded_food",
            "fdc_id in food_nutrient AND food_portion",
            "gram_weight > 0",
            "Word-boundary volume unit in modifier OR portion_description OR measure_unit.name "
            "(cup/cups whole words only; scripts/usda_volume_units.py)",
            "Food kept if ANY portion row passes 3+4",
        ],
        "count": [
            len(non_branded_food),
            len(food_mvp),
            int(mvp_portions["density_inferable"].sum()),
            len(food_signals),
            len(food_signals),
        ],
    }
)

portion_summary = pd.DataFrame(
    {
        "metric": [
            "food_mvp foods",
            "portion rows (mvp foods)",
            "rows with gram_weight > 0",
            "rows with volume measure",
            "rows density-inferable (gram_weight + volume)",
            "foods with >=1 density-inferable portion",
            "subset: modifier is cup or cups only",
        ],
        "count": [
            len(food_mvp),
            len(mvp_portions),
            int(mvp_portions["has_gram_weight"].sum()),
            int(mvp_portions["has_volume_measure"].sum()),
            int(mvp_portions["density_inferable"].sum()),
            len(food_signals),
            int(food_signals["cup_modifier_exact"].sum()),
        ],
    }
)
portion_summary["pct_of_food_mvp"] = (
    100.0 * portion_summary["count"] / len(food_mvp)
).round(2)

density_foods = food_mvp.merge(food_signals, on="fdc_id", how="left")
density_foods["has_density_portion"] = density_foods["fdc_id"].isin(food_signals["fdc_id"])

density_by_type = (
    density_foods.groupby("data_type", dropna=False)
    .agg(
        foods_in_mvp=("fdc_id", "count"),
        density_inferable_foods=("has_density_portion", "sum"),
        volume_via_modifier=("vol_modifier", "sum"),
        volume_via_description=("vol_description", "sum"),
        volume_via_measure_unit=("vol_measure_unit", "sum"),
        cup_modifier_only=("cup_modifier_exact", "sum"),
        avg_inferable_portions=("inferable_portion_rows", "mean"),
    )
    .assign(
        pct_density_inferable=lambda d: (
            100 * d["density_inferable_foods"] / d["foods_in_mvp"]
        ).round(1),
    )
    .sort_values("density_inferable_foods", ascending=False)
)

signal_mix = pd.DataFrame(
    {
        "signal": [
            "volume in modifier",
            "volume in portion_description",
            "volume in measure_unit.name",
            "modifier exactly cup or cups",
        ],
        "foods_with_signal": [
            int(food_signals["vol_modifier"].sum()),
            int(food_signals["vol_description"].sum()),
            int(food_signals["vol_measure_unit"].sum()),
            int(food_signals["cup_modifier_exact"].sum()),
        ],
    }
)
signal_mix["pct_of_density_inferable"] = (
    100.0 * signal_mix["foods_with_signal"] / len(food_signals)
).round(1)

print("How this group was determined (pipeline)")
display(criteria_steps)
print("\nPortion-level summary")
display(portion_summary)
print("\nVolume signal by field (not mutually exclusive)")
display(signal_mix)
print(
    "\nBy data_type: foods_in_mvp = non-branded with nutrients+portions; "
    "density_inferable_foods = passed steps 3-4; volume_via_* = foods where that field matched"
)
display(density_by_type)

How this group was determined (pipeline)


,step,rule,count
0,1. Universe,"food.csv, data_type != branded_food",101329
1,2. food_mvp,fdc_id in food_nutrient AND food_portion,13043
2,3. Portion row,gram_weight > 0,10945
3,4. Volume token,Word-boundary volume unit in modifier OR porti...,7700
4,5. Result,Food kept if ANY portion row passes 3+4,7700



Portion-level summary


,metric,count,pct_of_food_mvp
0,food_mvp foods,13043,100.00
1,portion rows (mvp foods),36679,281.22
2,rows with gram_weight > 0,36679,281.22
3,rows with volume measure,10945,83.91
4,rows density-inferable (gram_weight + volume),10945,83.91
5,foods with >=1 density-inferable portion,7700,59.04
6,subset: modifier is cup or cups only,1643,12.60



Volume signal by field (not mutually exclusive)


,signal,foods_with_signal,pct_of_density_inferable
0,volume in modifier,3322,43.1
1,volume in portion_description,4322,56.1
2,volume in measure_unit.name,56,0.7
3,modifier exactly cup or cups,1643,21.3



By data_type: foods_in_mvp = non-branded with nutrients+portions; density_inferable_foods = passed steps 3-4; volume_via_* = foods where that field matched


,foods_in_mvp,density_inferable_foods,volume_via_modifier,volume_via_description,volume_via_measure_unit,cup_modifier_only,avg_inferable_portions,pct_density_inferable
data_type,,,,,,,,
survey_fndds_food,5394,4322,0,4322,0,0,1.410227,80.1
sr_legacy_food,7533,3321,3321,0,0,1643,1.436615,44.1
foundation_food,116,57,1,0,56,0,1.385965,49.1


In [ ]:
### `food_mvp` funnel (local CSV → materialized view)

Sequential filters from all rows in `food.csv` down to **`usda.food_mvp`** (~7,700 foods): non-branded → nutrients → portions → density-inferable portion (`gram_weight > 0` + volume unit). Logic matches [`sql/06_create_food_mvp.sql`](../sql/06_create_food_mvp.sql).

Interactive chart below uses [Plotly `go.Funnel`](https://plotly.com/python/funnel-charts/). The funnel **starts at non-branded** (branded foods excluded upstream); each stage label still shows **% of all USDA foods** in `food.csv`. The table above lists every step including the full catalog.

In [5]:
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go


def _repo_root() -> Path:
    """Find Capstone root whether the kernel cwd is repo root or scratch/."""
    for cand in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (cand / "scripts" / "export_food_mvp_density_inferable.py").is_file():
            return cand
    raise FileNotFoundError(
        "Could not find scripts/export_food_mvp_density_inferable.py from cwd"
    )


if "ROOT" not in globals():
    ROOT = _repo_root()
_scripts_dir = str(ROOT / "scripts")
if _scripts_dir not in sys.path:
    sys.path.insert(0, _scripts_dir)

from export_food_mvp_density_inferable import (
    BRANDED_TYPE,
    DATA_DIR as USDA_DATA_DIR,
    collect_fdc_ids,
    density_inferable_fdc_ids,
)

if "DATA_DIR" not in globals():
    DATA_DIR = USDA_DATA_DIR

# Reuse session counts when CSV section was run; otherwise compute from files.
if "universe_ids" not in globals():
    if "load_non_branded_universe" in globals():
        non_branded_food, universe_ids = load_non_branded_universe()
    else:
        food = pd.read_csv(
            DATA_DIR / "food.csv",
            usecols=["fdc_id", "data_type"],
            dtype={"fdc_id": "int64", "data_type": "string"},
        )
        universe_ids = set(food.loc[food["data_type"] != BRANDED_TYPE, "fdc_id"])
if "nutrient_ids" not in globals():
    nutrient_ids = collect_fdc_ids(DATA_DIR / "food_nutrient.csv", universe_ids)
    portion_ids = collect_fdc_ids(DATA_DIR / "food_portion.csv", universe_ids)
    both_ids = nutrient_ids & portion_ids

if "branded_count" not in globals():
    branded_count = int(
        (
            pd.read_csv(DATA_DIR / "food.csv", usecols=["data_type"])["data_type"]
            == BRANDED_TYPE
        ).sum()
    )

total_foods = int(branded_count + len(universe_ids))
nutrient_and_portion = len(both_ids)

if "food_signals" in globals():
    food_mvp_count = len(food_signals)
else:
    food_mvp_count = len(density_inferable_fdc_ids(both_ids))

STAGE_LABELS = [
    "All USDA foods",
    "Non-branded",
    "Has nutrients",
    "Nutrients + portions",
    "food_mvp (density-inferable)",
]
STAGE_RULES = [
    "food.csv (all data_type values)",
    "data_type ≠ branded_food",
    "≥1 row in food_nutrient",
    "≥1 row in food_nutrient and food_portion",
    "≥1 portion: gram_weight > 0 + volume unit (sql/06_create_food_mvp.sql)",
]

funnel = pd.DataFrame(
    {
        "stage": STAGE_LABELS,
        "rule": STAGE_RULES,
        "count": [
            total_foods,
            len(universe_ids),
            len(nutrient_ids),
            nutrient_and_portion,
            food_mvp_count,
        ],
    }
)
funnel["pct_of_all_foods"] = (100 * funnel["count"] / total_foods).round(3)
funnel["pct_of_previous"] = (
    100 * funnel["count"] / funnel["count"].shift(1).fillna(funnel["count"].iloc[0])
).round(2)
funnel["dropped_from_previous"] = (
    funnel["count"].shift(1) - funnel["count"]
).fillna(0).astype(int)

display(
    funnel[
        [
            "stage",
            "rule",
            "count",
            "pct_of_all_foods",
            "pct_of_previous",
            "dropped_from_previous",
        ]
    ]
)

# Chart starts at non-branded; each segment still shows % of all USDA foods (not % of first box).
chart_funnel = funnel.iloc[1:].reset_index(drop=True)
FONT_FAMILY = "Inter, system-ui, sans-serif"
# Inside colored bands (dark/mid blue): light text. Outside on paper: dark text.
INSIDE_COLORS = ("#ffffff", "#f1f5f9", "#bfdbfe")  # title, count, % of all USDA
OUTSIDE_COLORS = ("#0f172a", "#0f172a", "#334155")

# Last two stages are narrow — labels sit outside the colored segments.
text_positions = ["inside", "inside", "outside", "outside"]

segment_text = []
for row, pos in zip(chart_funnel.itertuples(), text_positions):
    title_c, value_c, sub_c = INSIDE_COLORS if pos == "inside" else OUTSIDE_COLORS
    segment_text.append(
        f"<b style='color:{title_c}'>{row.stage}</b><br>"
        f"<span style='color:{value_c}'>{int(row.count):,}</span><br>"
        f"<span style='font-size:11px;color:{sub_c}'>"
        f"{row.pct_of_all_foods:.2f}% of all USDA foods</span>"
    )

label_font = dict(size=13, color=INSIDE_COLORS[0], family=FONT_FAMILY)
outside_font = dict(size=12, color=OUTSIDE_COLORS[0], family=FONT_FAMILY)

FUNNEL_COLORS = ["#1d4ed8", "#3b82f6", "#38bdf8", "#059669"]
FUNNEL_BORDERS = ["#93c5fd", "#bae6fd", "#a7f3d0", "#6ee7b7"]

fig = go.Figure(
    go.Funnel(
        y=chart_funnel["stage"],
        x=chart_funnel["count"],
        text=segment_text,
        textinfo="text",
        customdata=chart_funnel[
            ["rule", "dropped_from_previous", "pct_of_all_foods", "pct_of_previous"]
        ].values,
        textposition=text_positions,
        insidetextfont=label_font,
        outsidetextfont=outside_font,
        cliponaxis=False,
        constraintext="none",
        hovertemplate=(
            "<b>%{label}</b><extra></extra><br>"
            "Filter: %{customdata[0]}<br>"
            "Foods: <b>%{value:,}</b><br>"
            "%{customdata[2]:.3f}% of all USDA foods (%{customdata[3]:.2f}% of previous stage)<br>"
            "Dropped from prior: %{customdata[1]:,}"
        ),
        marker=dict(
            color=FUNNEL_COLORS,
            line=dict(color=FUNNEL_BORDERS, width=[2, 2, 2, 3]),
        ),
        connector=dict(
            line=dict(color="rgba(148, 163, 184, 0.55)", width=1.5),
            fillcolor="rgba(226, 232, 240, 0.35)",
        ),
        opacity=0.95,
    )
)

non_branded_pct = 100 * len(universe_ids) / total_foods
retention_pct = 100 * food_mvp_count / total_foods
fig.update_layout(
    title=dict(
        text=(
            "<b>Non-branded USDA food → food_mvp filtering funnel</b><br>"
            f"<span style='font-size:13px;color:#64748b'>"
            f"Universe: {len(universe_ids):,} non-branded foods "
            f"({non_branded_pct:.2f}% of {total_foods:,} in food.csv) → "
            f"{food_mvp_count:,} density-inferable ({retention_pct:.2f}% of all USDA foods)</span>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=18, family="Inter, system-ui, sans-serif", color="#0f172a"),
    ),
    funnelmode="stack",
    template="plotly_white",
    width=960,
    height=580,
    margin=dict(l=32, r=160, t=96, b=32),
    paper_bgcolor="#f8fafc",
    plot_bgcolor="#f8fafc",
    font=dict(family="Inter, system-ui, sans-serif", color="#334155", size=12),
    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#cbd5e1",
        font=dict(family="Inter, system-ui, sans-serif", size=12),
    ),
)

fig.show()

,stage,rule,count,pct_of_all_foods,pct_of_previous,dropped_from_previous
0,All USDA foods,food.csv (all data_type values),2101279,100.000,100.00,0
1,Non-branded,data_type ≠ branded_food,101329,4.822,4.82,1999950
2,Has nutrients,≥1 row in food_nutrient,89260,4.248,88.09,12069
3,Nutrients + portions,≥1 row in food_nutrient and food_portion,13043,0.621,14.61,76217
4,food_mvp (density-inferable),≥1 portion: gram_weight > 0 + volume unit (sql...,7700,0.366,59.04,5343
